### Libraries 

In [1]:
!pip install openai

In [2]:
import pandas as pd
import numpy as np
# from tqdm import tqdm
import random
import time
import csv
import json
import openai
import os
from tqdm import tqdm
import time

### Dataset loading and analysis

#### Arguments

In [3]:
arguments = pd.read_csv("arguments.csv")
arguments.head()

,Argument ID,Part,Usage,Conclusion,Stance,Premise
0,A01001,usa,train,Entrapment should be legalized,in favor of,if entrapment can serve to more easily capture...
1,A01002,usa,train,We should ban human cloning,in favor of,we should ban human cloning as it will only ca...
2,A01003,usa,train,We should abandon marriage,against,marriage is the ultimate commitment to someone...
3,A01004,usa,train,We should ban naturopathy,against,it provides a useful income for some people
4,A01005,usa,train,We should ban fast food,in favor of,fast food should be banned because it is reall...


In [4]:
count = 0
for conclusion, stance, premise in arguments[["Conclusion", "Stance", "Premise"]].tail(10).itertuples(index=False, name=None):
    if count == 3:
        break
    print("C: {}, S: {}, P: {}".format(conclusion, stance, premise))
    count+=1
    

C: Reservations in India should be based on economic status, S: against, P: In India, it's easy to hide some income resources and to get a false income certificate. Though the situations are changing for the better, it is not that difficult.
C: There should be a ban on Bandhs, S: in favor of, P: Industries are also one of the worst hits. Investors may lose interest to invest in manufacturing sector as bandh by anyone results in the loss to them.
C: There should be a ban on Bandhs, S: in favor of, P: Though right to protest is a fundamental right, people do not have right to take away the fundamental rights of others such as going to their work, doing their personal works etc. Not everyone is against to the changes by government. Some people may support it and may not support the ban. Forcing them to stay at home is undemocratic.


#### Subsetting arguments from USA - case of study

In [5]:
arguments_usa = arguments[arguments["Part"] == "usa"]
print("Dataframe with USA instances has {} arguments".format(arguments_usa.shape[0]))

Dataframe with USA instances has 5020 arguments


#### Values

In [6]:
import json

with open("values.json", "r", encoding="utf-8") as f:
    values_dict = json.load(f)
values_l1_in_json = {item["name"] for item in values_dict["values"]} # value names in json file - level 1

ground_truth = pd.read_csv("labels-level1.csv")
df_value_columns = set(ground_truth.columns[1:])  # skip first column (Argument ID)

# all values in json correspond to the 54 value categories from the ground truth


In [7]:
def build_values_prompt_text(values_dict):
    lines = []

    for v in values_dict["values"]:
        name = v["name"]
        vid = v["id"]
        desc = "; ".join(v["descriptions"]) 

        line = f'{name} - {desc}'
        lines.append(line)

    # return '\n'.join(lines)
    return lines

VALUES_LIST = build_values_prompt_text(values_dict)

# len(VALUES_LIST)

## Baseline Predictor of Values (1000 arguments)

In [8]:
arguments_503 = pd.read_csv("arguments_503.csv")
# we want to predict the same 503 arguments as BERT and SVM from the reference paper
l_503 = arguments_503['Argument ID'].to_list()
# some more arguments that are not part of the 503 arguments above, to analyse a bigger sample
l_497 = (
    arguments_usa.loc[~arguments_usa["Argument ID"].isin(l_503), "Argument ID"]
    .head(497)
    .tolist()
)
l_1000 = l_503 + l_497
print(f"Hay un total de {len(l_1000)} argumentos a analizar. Sin repeticiones, {len(set(l_1000))}.")
 # comprobar que no hay repeticiones en los argumentos

arguments_1000 = arguments_usa.loc[
    arguments_usa["Argument ID"].isin(l_503) |
    arguments_usa["Argument ID"].isin(l_497)
]

arguments_1000

Hay un total de 1000 argumentos a analizar. Sin repeticiones, 1000.


,Argument ID,Part,Usage,Conclusion,Stance,Premise
0,A01001,usa,train,Entrapment should be legalized,in favor of,if entrapment can serve to more easily capture...
1,A01002,usa,train,We should ban human cloning,in favor of,we should ban human cloning as it will only ca...
2,A01003,usa,train,We should abandon marriage,against,marriage is the ultimate commitment to someone...
3,A01004,usa,train,We should ban naturopathy,against,it provides a useful income for some people
4,A01005,usa,train,We should ban fast food,in favor of,fast food should be banned because it is reall...
...,...,...,...,...,...,...
5015,A25476,usa,test,We should oppose collectivism,in favor of,opposing collectivism is the right thing to do...
5016,A25480,usa,test,We should limit judicial activism,against,activist judges are an important part of bring...
5017,A25486,usa,test,We should subsidize student loans,in favor of,graduates should not have to start their caree...
5018,A25493,usa,test,We should oppose collectivism,against,collectivism is society working for the benefi...


##### Check if the LLM call works

In [9]:
# import base_url and api_key from private files 
from pathlib import Path

file_path = Path('Cognitive-Bias-and-Heuristics-in-Value-Identification\baseline_predictor.ipynb').resolve().parent.parent / "keys.txt"

with open(file_path, "r", encoding="utf-8") as f:
    first_two_lines = [next(f).rstrip("\n") for _ in range(2)]

base_url, api_key = first_two_lines[0].strip(), first_two_lines[1].strip()

# !pip install openai

from openai import OpenAI
def crear_cliente():
    client = OpenAI(
        base_url=base_url,
        api_key=api_key, 
    )
    return client
 
def call_llm(client, model, prompt):
    response = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model=model
    )
    return response.choices[0].message.content


In [10]:
client = crear_cliente()
call_llm(client, 'gpt-oss-120b', 'qué es la IA generativa')

'**IA generativa** (inteligencia artificial generativa) es una rama de la IA que se centra en crear o “generar” contenido nuevo que antes no existía, a partir de patrones aprendidos de datos de entrenamiento. En lugar de solo reconocer, clasificar o predecir información (como hace la IA discriminativa), la IA generativa produce textos, imágenes, audio, video, código, diseños, etc., que pueden ser indistinguibles de los creados por humanos.\n\n---\n\n## 1. ¿Cómo funciona?\n\n1. **Modelo de aprendizaje profundo**  \n   - Se entrena una red neuronal (por ejemplo, Transformers, Redes Generativas Adversariales *GAN*, Modelos de Difusión, VAE, etc.) con grandes volúmenes de datos representativos del dominio que se quiere generar (texto, fotos, música…).\n\n2. **Captura de patrones estadísticos**  \n   - La red aprende la distribución probatoria subyacente de esos datos; esencialmente “comprende” cómo suelen combinarse los elementos (palabras, píxeles, notas).\n\n3. **Muestreo / generación** 

#### Prompt Engineering and Experimentation

In [11]:
# len(['Be creative', 'Be curious', 'Have freedom of thought', 'Be choosing own goals', 'Be independent', 'Have freedom of action', 'Have privacy', 'Have an exciting life', 'Have a varied life', 'Be daring', 'Have pleasure', 'Be ambitious', 'Have success', 'Be capable', 'Be intellectual', 'Be courageous', 'Have influence', 'Have the right to command', 'Have wealth', 'Have social recognition', 'Have a good reputation', 'Have a sense of belonging', 'Have good health', 'Have no debts', 'Be neat and tidy', 'Have a comfortable life', 'Have a safe country', 'Have a stable society', 'Be respecting traditions', 'Be holding religious faith', 'Be compliant', 'Be self-disciplined', 'Be behaving properly', 'Be polite', 'Be honoring elders', 'Be humble', 'Have life accepted as is', 'Be helpful', 'Be honest', 'Be forgiving', 'Have the own family secured', 'Be loving', 'Be responsible', 'Have loyalty towards friends', 'Have equality', 'Be just', 'Have a world at peace', 'Be protecting the environment', 'Have harmony with nature', 'Have a world of beauty', 'Be broadminded', 'Have the wisdom to accept others', 'Be logical', 'Have an objective view'])

In [12]:
VALUE_PREDICTION_PROMPT = """
# ROLE & GOAL
You are an expert annotator. Your task is to determine which of 54 human values could reasonably serve as a justification for a given argument.

# INPUT STRUCTURE
You will receive:
- STANCE: "for" or "against"
- CONCLUSION: The main claim (e.g., "Social media should be banned")
- PREMISE: The reasoning provided (e.g., "It spreads misinformation")
- VALUES_LIST: A fixed list of 54 values with short descriptions (provided below)

# SELECTION GUIDELINES:
- Select for the given argument which of 54 justifications one could provide for it. 
- Typically, one could provide at least 1 and not more than 5 of these justifications for an argument. If you would select more than 10 justifications for an argument, reduce your selection to the most fitting ones.
- Make sure you understand the examples given.
- Read the argument and justification. Select 1 (someone could provide the justification for the argument, even if you may disagree) or 0 (the justification makes no sense for the argument).

# WORKFLOW
1. Imagine someone is arguing "{STANCE}" "{CONCLUSION}" by saying: "{PREMISE}"
2. For each value in the VALUES_LIST (in fixed order), If asked 'Why is this good?', might this be their justification? "Because it is good to "value"".
3. Assign 1 or 0 accordingly.


# CRITICAL FORMATTING REQUIREMENTS
- Output **must** be a single Python dictionary with exactly 54 key-value pairs.
- Keys **must** be the exact value names as given in the VALUES_LIST, in the **exact same order**.
- Values **must** be integers: `1` or `0` (no quotes, no booleans).
- **No** additional text, explanations, markdown, or code fences before or after the dictionary.
- **No** skipped keys, no reordered keys, no extra keys.


# EXAMPLES:
Example arguments against "Social media should be banned". - **Do** use the provided examples for each value to calibrate your understanding, but do not over‑apply them mechanically.

- Argument: We have to be honest. Social media does not make people polite. But it makes our lives easier and more interesting.	
- Select all justifications one could provide: (✔ have a comfortable life) (from "easier lives"), (✔ have pleasure) (also from "easier lives"), (✔  ahaven exciting life) (from "more interesting"), (✔ have a varied life) (also from "more interesting"). But do not select justifications for concessions ((✖ be polite)) or empty phrases ((✖ be honest), (✖ be logical), (✖ have an objective view) for "We have to be honest").
Example arguments in favor of "Social media should be banned".
- Argument: Through social media people can spread biased opinions on topics or misinform the general public.	
- Use the examples for each justification to get a better understanding of the justifications ((✔ have freedom of thought) from reduced misleading influence on people's thoughts). But do not select justifications only because they are connected to the topic in general ((✖ have privacy) for the general threat of social media to privacy: it is not mentioned here).


# VALUES_LIST (in fixed order)
 "{VALUES_LIST}"

 
# NOW, PERFORM THE TASK FOR THE FOLLOWING ARGUMENT:
Argument: "{STANCE}" "{CONCLUSION}" - Premise: "{PREMISE}"
 
# EXAMPLE OUTPUT (illustrative, not for this input)
Example output as dictionary:
{{'Be creative':1, 'Be curious':0, 'Have freedom of thought':0, 'Be choosing own goals':0,..., 'Have the wisdom to accept others':1, 'Be logical':0, 'Have an objective view':0}}
"""

In [13]:
OLD_VALUE_PREDICTION_PROMPT = """
INSTRUCTIONS:
- Select for the given argument which of 54 justifications one could provide for it. 
- Typically, one could provide at least 1 and not more than 5 of these justifications for an argument. If you would select more than 10 justifications for an argument, reduce your selection to the most fitting ones.
- Make sure you understand the examples given.
- Read the argument and justification. Select 1 (someone could provide the justification for the argument, even if you may disagree) or 0 (the justification makes no sense for the argument).

EXAMPLES:
Example arguments against "Social media should be banned".

- Argument: We have to be honest. Social media does not make people polite. But it makes our lives easier and more interesting.	
- Select all justifications one could provide: (✔ have a comfortable life) (from "easier lives"), (✔ have pleasure) (also from "easier lives"), (✔  ahaven exciting life) (from "more interesting"), (✔ have a varied life) (also from "more interesting"). But do not select justifications for concessions ((✖ be polite)) or empty phrases ((✖ be honest), (✖ be logical), (✖ have an objective view) for "We have to be honest").
Example arguments in favor of "Social media should be banned".
- Argument: Through social media people can spread biased opinions on topics or misinform the general public.	
- Use the examples for each justification to get a better understanding of the justifications ((✔ have freedom of thought) from reduced misleading influence on people's thoughts). But do not select justifications only because they are connected to the topic in general ((✖ have privacy) for the general threat of social media to privacy: it is not mentioned here).

TASK:
Select for the given argument which of 54 justifications one could provide for it. 

Imagine someone is arguing "{STANCE}" "{CONCLUSION}" by saying: "{PREMISE}"
If asked 'Why is this good?', might this be their justification? "Because it is good to "VALUE"".

Complete this task for each of the justifications, that is, each of the 54 human values with their descriptions in the following list.

Justifications (in fixed order): "{VALUES_LIST}"


Return format:
- Return a dictionary of EXACTLY 54 keys and 54 values. Keys correspond to each of the justifications, following the fixed order. The dictionary values must be an integer: 0 or 1.
    - 1 = YES (someone could provide the justification for the argument)
    - 0 = NO (the justification makes no sense for the argument)
- The order MUST match exactly the order of the VALUES list: ['Be creative', 'Be curious', 'Have freedom of thought', 'Be choosing own goals', 'Be independent', 'Have freedom of action', 'Have privacy', 'Have an exciting life', 'Have a varied life', 'Be daring', 'Have pleasure', 'Be ambitious', 'Have success', 'Be capable', 'Be intellectual', 'Be courageous', 'Have influence', 'Have the right to command', 'Have wealth', 'Have social recognition', 'Have a good reputation', 'Have a sense of belonging', 'Have good health', 'Have no debts', 'Be neat and tidy', 'Have a comfortable life', 'Have a safe country', 'Have a stable society', 'Be respecting traditions', 'Be holding religious faith', 'Be compliant', 'Be self-disciplined', 'Be behaving properly', 'Be polite', 'Be honoring elders', 'Be humble', 'Have life accepted as is', 'Be helpful', 'Be honest', 'Be forgiving', 'Have the own family secured', 'Be loving', 'Be responsible', 'Have loyalty towards friends', 'Have equality', 'Be just', 'Have a world at peace', 'Be protecting the environment', 'Have harmony with nature', 'Have a world of beauty', 'Be broadminded', 'Have the wisdom to accept others', 'Be logical', 'Have an objective view']
- Do NOT skip any value and do NOT add text, explanation, or formatting
- the output MUST BE a DICTIONARY 

Example output as dictionary:
{'Be creative':1, 'Be curious':0, 'Have freedom of thought':0, 'Be choosing own goals':0,..., 'Have the wisdom to accept others':1, 'Be logical':0, 'Have an objective view':0}
(with the length of the list being EXACTLY 54 and the order matching the VALUES list)
"""


In [14]:
def run_experiment(client, model, arguments, output_path):
    if os.path.exists(output_path):
        existing_df = pd.read_csv(output_path)
        done_ids = set(existing_df["Argument ID"].tolist())
        results = existing_df.values.tolist()
        print(f"Resuming... {len(done_ids)} arguments already processed (out of {len(arguments)}).")
        if len(done_ids) == len(arguments):
            return True
    else:
        done_ids = set()
        results = []

    segons_llista = []

    for _, row in tqdm(arguments.iterrows(), total=len(arguments)):
        arg_id = row["Argument ID"]

        if arg_id in done_ids:
            continue

        start = time.time()
        
        prompt = VALUE_PREDICTION_PROMPT.format(
            CONCLUSION=row["Conclusion"],
            STANCE=row["Stance"],
            PREMISE=row["Premise"],
            VALUES_LIST=VALUES_LIST
        )

        vector = call_llm(client, model, prompt)

        end = time.time()
        segons = end - start
        segons_llista.append(segons)

        results.append([arg_id] + [str(vector)] + [segons])

        df = pd.DataFrame(
            results,
            columns=["Argument ID"] + ["values"] + ["seconds"]
        )

        df.to_csv(output_path, index=False)

    return pd.DataFrame(results, columns=["Argument ID"] + ["values"] + ["seconds"])

In [15]:
VALUES_LIST

['Be creative - allowing for more creativity or imagination; being more creative; fostering creativity; promoting imagination',
 'Be curious - being the more interesting option; fostering curiosity; making people more keen to learn; promoting discoveries; sparking interest',
 "Have freedom of thought - allowing people to figure things out on their own; allowing people to make up their mind; resulting in less censorship; resulting in less influence on people's thoughts",
 'Be choosing own goals - allowing people to choose what is best for them; allowing people to decide on their life; allowing people to follow their dreams',
 'Be independent - allowing people to plan on their own; resulting in fewer times people have to ask for consent',
 'Have freedom of action - allowing people to be self-determined; allowing people to do things even though this may hurt them in the long run; resulting in more times people can do what they want',
 'Have privacy - allowing for private spaces; allowing 

#### LLM selection

Si el objetivo es seguir instrucciones de texto (instruction following), sin necesidad de programar o analizar imágenes, el orden aproximado sería:
* **Qwen3.6-35B-A3B**	⭐⭐⭐⭐⭐	Uno de los mejores modelos abiertos actuales para seguir instrucciones, razonamiento y conversación. https://huggingface.co/Qwen/Qwen3.6-35B-A3B // https://qwen.ai/blog?id=qwen3.6-35b-a3b

* **Llama 3.3 70B**	⭐⭐⭐⭐⭐	Excelente capacidad conversacional e instruction following. Muy cercano a Qwen, aunque suele ser algo menos preciso en tareas complejas. https://ollama.com/library/llama3.3:70b

* **GPT-OSS-120B**	⭐⭐⭐⭐☆	Muy potente por tamaño (120B), pero en la práctica no siempre supera a Qwen 3.6 en adherencia estricta a instrucciones. https://openai.com/es-ES/index/introducing-gpt-oss/

In [16]:
# llms = ['gpt-oss-120b', 'llama3.3:70b', 'Qwen3.6-35B-A3B']
llms = ['gpt-oss-120b']
output_csv = ['results/baseline_gpt_oss_1000.csv','results/baseline_llama_1000.csv','results/baseline_qwen_1000.csv']

#### Run Experiments

In [17]:
client = crear_cliente()
for model, output in zip(llms, output_csv):
    results = run_experiment(client, model, arguments_1000, output)

Resuming... 1000 arguments already processed (out of 1000).


#### Handle baseline prediction ERRORS
44 INSTANCES HAVE NOT FOLLOWED THE INSTRUCTIONS GIVEN IN THE PROMPT, REPEAT EXPERIMENTS

In [18]:
erroneous_arguments = ['A01006', 'A01010', 'A01017', 'A02001', 'A03004', 'A05087', 'A07014', 'A07045', 'A07066', 'A07082', 'A07087', 'A08011', 'A09016', 'A09022', 'A09031', 'A09066', 'A12047', 'A12075', 'A12094', 'A12109', 'A12111', 'A06017', 'A07017', 'A12138', 'A12285', 'A12361', 'A18457', 'A18464', 'A18495', 'A19082', 'A19361', 'A20490', 'A21424', 'A21448', 'A21477', 'A22037', 'A22094', 'A22228', 'A22476', 'A23244', 'A24111', 'A24141', 'A24159', 'A25309']

In [19]:
args_errors_df = arguments_1000[arguments_1000["Argument ID"].isin(erroneous_arguments)]
print(args_errors_df.shape)
args_errors_df.head()

(44, 6)


,Argument ID,Part,Usage,Conclusion,Stance,Premise
5,A01006,usa,train,We should end the use of economic sanctions,against,sometimes economic sanctions are the only thin...
9,A01010,usa,train,We should prohibit school prayer,against,it should be allowed if the student wants to p...
16,A01017,usa,train,We should end affirmative action,in favor of,affirmative action is no longer necessary as m...
20,A02001,usa,train,Payday loans should be banned,in favor of,payday loans create a more impoverished societ...
43,A03004,usa,train,We should abolish the three-strikes laws,against,this law keeps communities safe by not letting...


In [20]:
model = 'gpt-oss-120b'
output = 'results/baseline_gpt_oss_44.csv'
results = run_experiment(client, model, args_errors_df, output)

Resuming... 44 arguments already processed (out of 44).


Dels 44 per corregir, només 5 no han pogut ser solventats. En els cinc casos, es torna el VALUE list, amb els noms dels valors i les descripcions, tal cual. 

5 INSTANCES HAVE NOT FOLLOWED THE INSTRUCTIONS GIVEN IN THE PROMPT, REPEAT EXPERIMENTS

In [21]:
errors_last = ['A05087', 'A12047', 'A12361','A18457','A21424']
args_errors_df1 = arguments_1000[arguments_1000["Argument ID"].isin(errors_last)]
print(args_errors_df1.shape)
args_errors_df1.head()

(5, 6)


,Argument ID,Part,Usage,Conclusion,Stance,Premise
153,A05087,usa,train,We should end mandatory retirement,in favor of,We should end mandatory retirement because it ...
423,A12047,usa,train,Payday loans should be banned,in favor of,payday lenders charge exorbitant interest rate...
4593,A12361,usa,test,We should subsidize student loans,against,tax payers shouldn't have to pay for other peo...
4659,A18457,usa,test,Homeopathy brings more harm than good,against,homeopathy has been proven to have a successfu...
4813,A21424,usa,test,We should subsidize student loans,in favor of,by subsidizing student loans we can remove one...


In [ ]:
model = 'gpt-oss-120b'
output = 'results/baseline_gpt_oss_5.csv'
results = run_experiment(client, model, args_errors_df1, output)

Resuming... 1 arguments already processed (out of 5).


  0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 5/5 [02:56<00:00, 35.29s/it]


### Merge dataset without errors

In [27]:
df1_1000 = pd.read_csv('results/baseline_gpt_oss_1000.csv')
df2_44 = pd.read_csv('results/baseline_gpt_oss_44.csv')
df3_5 = pd.read_csv('results/baseline_gpt_oss_5.csv')

final_df_baseline = (
    df3_5.set_index("Argument ID")
       .combine_first.set_index(df2_44("Argument ID"))
       .combine_first(df1_1000.set_index("Argument ID"))
       .reset_index()
)

final_df_baseline = (
    pd.concat([df1_1000, df2_44, df3_5], ignore_index=True)
      .drop_duplicates(subset="Argument ID", keep="last")
)


final_df_baseline.to_csv('results/baseline_FINAL.csv', index=False)



AttributeError: 'function' object has no attribute 'set_index'

## Cognitive Bias Predictor of Values (1000 arguments)

In [ ]:
arguments_503.head()

### Cognitive Bias Prompt Experts

##### Framing Effect Prompt

##### Cognitive Bias Prompt

##### Hasty Generalisation Prompt

##### Causal Oversimplification Prompt

In [ ]:
CAUSAL_OVERSIMPLIFICATION_PROMPT = """

# ROLE & GOAL
You are an expert in cognitive biases and human reasoning. Your task is to determine which of 54 universal human values could serve as a justification for a given argument, but you must analyze this through the lens of **causal oversimplification bias**—the tendency to attribute complex outcomes to a single, simple cause while ignoring other contributing factors.

# INPUT STRUCTURE
You will receive:
- STANCE: "for" or "against"
- CONCLUSION: The main claim (e.g., "Social media should be banned")
- PREMISE: The reasoning provided (e.g., "It spreads misinformation")
- VALUES_LIST: A fixed list of 54 values with short descriptions (provided below)

# TASK DEFINITION
For each value in the VALUES_LIST, decide if someone *could* use that value to justify the argument, **considering how causal oversimplification might shape or distort that justification**.

**Decision rule:** Imagine the arguer is asked: "Why is this good/important?" Would answering "Because it promotes [VALUE]" be a coherent justification that **could plausibly arise from oversimplified causal reasoning**? Consider:

- **Single-cause attribution:** Does the premise attribute a complex problem to one cause (e.g., "Social media causes misinformation") while ignoring other factors?
- **Silver bullet thinking:** Does the premise imply that banning social media will *solve* the problem completely, as if it's the only cause?
- **Linear causality:** Does the premise assume a direct, unidirectional cause-effect relationship (A → B) when it's actually cyclical or multi-directional?
- **Omission of moderators/mediators:** Does the premise ignore that the effect only occurs under certain conditions?
- **Confusing correlation with causation:** Does the premise treat association as proof of causation?
- **Solution-focused values:** Does the oversimplification lead to an overemphasis on values that represent *solutions* (safety, stability, justice, health) while suppressing values that address *root causes* (intellectual, broadminded, wisdom, curiosity)?

**CRITICAL:** Your task is NOT to determine if the justification is *logically sound* or *factually correct*. Your task is to determine if **a cognitively biased human could genuinely use this value as a justification**, especially when their reasoning is oversimplified.

If yes → 1, if no → 0.

# SELECTION GUIDELINES (BIAS-AWARE)
- **Typical arguments** may justify with 1–5 values, but oversimplified causal reasoning can inflate this to 5–8 values by making the problem seem simpler and the solution more effective. If you identify >10 plausible values, select only the **most cognitively plausible** ones—those most likely to emerge from oversimplified causal reasoning.
- **Do NOT select** values that are:
  - Merely generic or tautological (e.g., "be logical", "have an objective view") **unless** the argument explicitly invokes logic/objectivity in a biased way (e.g., "I'm being logical" as a defense mechanism).
  - Concessions or counterarguments, **unless** the arguer is using them in a biased manner (e.g., acknowledging a weakness but downplaying it via motivated reasoning).
  - Topically associated but not actually used in the reasoning, **unless** the association is so strong that availability bias would make it salient.
- **DO select** values that:
  - Represent *solutions* to the oversimplified problem (e.g., safety, stability, justice, health).
  - Are emotionally charged or vivid (availability heuristic, affect heuristic).
  - Reinforce the arguer's pre-existing beliefs (confirmation bias, belief perseverance).
  - Avoid losses or threats more than they seek gains (loss aversion, negativity bias).
  - Protect social status, reputation, or face (social desirability bias, self-enhancement).
- **Do** consider that multiple biases can amplify or override each other (e.g., causal oversimplification + confirmation bias → strong selective value endorsement for simplistic solutions).

# CAUSAL OVERSIMPLIFICATION-SPECIFIC WORKFLOW
1. Read the argument: "{STANCE}" "{CONCLUSION}" by saying: "{PREMISE}"
2. **Identify potential causal oversimplifications** that could influence how this argument is formulated and justified.
   - What complex phenomenon is being reduced to a single cause?
   - What other contributing factors are being ignored?
   - Is the premise treating correlation as causation?
   - Is the arguer suggesting a "silver bullet" solution?
3. For each value in the VALUES_LIST (in fixed order), ask: *"Could this value be a genuine justification for this specific premise, **given the causal oversimplifications identified?** "*
   - Would an oversimplifying reasoner latch onto this value as a *solution* to the problem?
   - Does this value represent a *simplistic fix* rather than a *nuanced approach*?
   - Is this value being selected because the arguer believes the problem is simpler than it actually is?
4. Assign 1 or 0 accordingly.

# CRITICAL FORMATTING REQUIREMENTS
- Output **must** be a single Python dictionary with exactly 54 key‑value pairs.
- Keys **must** be the exact value names as given in the VALUES_LIST, in the **exact same order**.
- Values **must** be integers: `1` or `0` (no quotes, no booleans).
- **No** additional text, explanations, markdown, or code fences before or after the dictionary.
- **No** skipped keys, no reordered keys, no extra keys.

# VALUES_LIST (in fixed order)
['Be creative', 'Be curious', 'Have freedom of thought', 'Be choosing own goals', 'Be independent', 'Have freedom of action', 'Have privacy', 'Have an exciting life', 'Have a varied life', 'Be daring', 'Have pleasure', 'Be ambitious', 'Have success', 'Be capable', 'Be intellectual', 'Be courageous', 'Have influence', 'Have the right to command', 'Have wealth', 'Have social recognition', 'Have a good reputation', 'Have a sense of belonging', 'Have good health', 'Have no debts', 'Be neat and tidy', 'Have a comfortable life', 'Have a safe country', 'Have a stable society', 'Be respecting traditions', 'Be holding religious faith', 'Be compliant', 'Be self-disciplined', 'Be behaving properly', 'Be polite', 'Be honoring elders', 'Be humble', 'Have life accepted as is', 'Be helpful', 'Be honest', 'Be forgiving', 'Have the own family secured', 'Be loving', 'Be responsible', 'Have loyalty towards friends', 'Have equality', 'Be just', 'Have a world at peace', 'Be protecting the environment', 'Have harmony with nature', 'Have a world of beauty', 'Be broadminded', 'Have the wisdom to accept others', 'Be logical', 'Have an objective view']

# EXAMPLE OUTPUT (illustrative, not for this input)
{'Be creative': 1, 'Be curious': 0, 'Have freedom of thought': 0, 'Be choosing own goals': 0, 'Be independent': 0, 'Have freedom of action': 0, 'Have privacy': 0, 'Have an exciting life': 1, 'Have a varied life': 1, 'Be daring': 0, 'Have pleasure': 1, 'Be ambitious': 0, 'Have success': 0, 'Be capable': 0, 'Be intellectual': 0, 'Be courageous': 0, 'Have influence': 0, 'Have the right to command': 0, 'Have wealth': 0, 'Have social recognition': 0, 'Have a good reputation': 0, 'Have a sense of belonging': 0, 'Have good health': 0, 'Have no debts': 0, 'Be neat and tidy': 0, 'Have a comfortable life': 1, 'Have a safe country': 0, 'Have a stable society': 0, 'Be respecting traditions': 0, 'Be holding religious faith': 0, 'Be compliant': 0, 'Be self-disciplined': 0, 'Be behaving properly': 0, 'Be polite': 0, 'Be honoring elders': 0, 'Be humble': 0, 'Have life accepted as is': 0, 'Be helpful': 0, 'Be honest': 0, 'Be forgiving': 0, 'Have the own family secured': 0, 'Be loving': 0, 'Be responsible': 0, 'Have loyalty towards friends': 0, 'Have equality': 0, 'Be just': 0, 'Have a world at peace': 0, 'Be protecting the environment': 0, 'Have harmony with nature': 0, 'Have a world of beauty': 0, 'Be broadminded': 0, 'Have the wisdom to accept others': 0, 'Be logical': 0, 'Have an objective view': 0}

# CAUSAL OVERSIMPLIFICATION EXPERT CONTEXT (Mental Model)
- **Your expertise includes:** Single-cause attribution, silver bullet thinking, linear causality, correlation vs. causation, omission of moderators/mediators, reductionist reasoning, and the tendency to oversimplify complex social, technological, or psychological phenomena.
- **Your mission:** Identify values that a *causally oversimplifying* human would select, not values that a *rational* human would select. The arguer may reduce complex problems to simple causes, may ignore systemic factors, may treat correlation as causation, or may genuinely believe that a single intervention will solve everything.

# TASK COMPLETION CHECKLIST
- [ ] Read the argument carefully
- [ ] Identify causal oversimplifications in the premise
- [ ] For each of the 54 values, decide if it could be a justification given oversimplified reasoning
- [ ] Output EXACTLY 54 key-value pairs in the correct order
- [ ] Ensure output is a dictionary with integer values (0 or 1)
- [ ] Do NOT add any text before or after the dictionary

# NOW, PERFORM THE TASK FOR THE FOLLOWING ARGUMENT:
Argument: "{STANCE}" "{CONCLUSION}" - Premise: "{PREMISE}"

"""

In [ ]:
CAUSAL_OVERSIMPLIFICATION_PROMPT = """

# ROLE & GOAL
You are an expert in cognitive biases and human reasoning. Your task is to determine which of 54 universal human values could serve as a justification for a given argument, but you must analyze this through the lens of **causal oversimplification bias**—the tendency to attribute complex outcomes to a single, simple cause while ignoring other contributing factors.

# INPUT STRUCTURE
You will receive:
- STANCE: "for" or "against"
- CONCLUSION: The main claim (e.g., "Social media should be banned")
- PREMISE: The reasoning provided (e.g., "It spreads misinformation")
- VALUES_LIST: A fixed list of 54 values with short descriptions (provided below)

# TASK DEFINITION
For each value in the VALUES_LIST, decide if someone *could* use that value to justify the argument, **considering how causal oversimplification might shape or distort that justification**.

**Decision rule:** Imagine the arguer is asked: "Why is this good/important?" Would answering "Because it promotes [VALUE]" be a coherent justification that **could plausibly arise from oversimplified causal reasoning**? Consider:

- **Single-cause attribution:** Does the premise attribute a complex problem to one cause (e.g., "Social media causes misinformation") while ignoring other factors?
- **Silver bullet thinking:** Does the premise imply that banning social media will *solve* the problem completely, as if it's the only cause?
- **Linear causality:** Does the premise assume a direct, unidirectional cause-effect relationship (A → B) when it's actually cyclical or multi-directional?
- **Omission of moderators/mediators:** Does the premise ignore that the effect only occurs under certain conditions?
- **Confusing correlation with causation:** Does the premise treat association as proof of causation?
- **Solution-focused values:** Does the oversimplification lead to an overemphasis on values that represent *solutions* (safety, stability, justice, health) while suppressing values that address *root causes* (intellectual, broadminded, wisdom, curiosity)?

**CRITICAL:** Your task is NOT to determine if the justification is *logically sound* or *factually correct*. Your task is to determine if **a cognitively biased human could genuinely use this value as a justification**, especially when their reasoning is oversimplified.

If yes → 1, if no → 0.

# SELECTION GUIDELINES (BIAS-AWARE)
- **Typical arguments** may justify with 1–5 values, but oversimplified causal reasoning can inflate this to 5–8 values by making the problem seem simpler and the solution more effective. If you identify >10 plausible values, select only the **most cognitively plausible** ones—those most likely to emerge from oversimplified causal reasoning.
- **Do NOT select** values that are:
  - Merely generic or tautological (e.g., "be logical", "have an objective view") **unless** the argument explicitly invokes logic/objectivity in a biased way (e.g., "I'm being logical" as a defense mechanism).
  - Concessions or counterarguments, **unless** the arguer is using them in a biased manner (e.g., acknowledging a weakness but downplaying it via motivated reasoning).
  - Topically associated but not actually used in the reasoning, **unless** the association is so strong that availability bias would make it salient.
- **DO select** values that:
  - Represent *solutions* to the oversimplified problem (e.g., safety, stability, justice, health).
  - Are emotionally charged or vivid (availability heuristic, affect heuristic).
  - Reinforce the arguer's pre-existing beliefs (confirmation bias, belief perseverance).
  - Avoid losses or threats more than they seek gains (loss aversion, negativity bias).
  - Protect social status, reputation, or face (social desirability bias, self-enhancement).
- **Do** consider that multiple biases can amplify or override each other (e.g., causal oversimplification + confirmation bias → strong selective value endorsement for simplistic solutions).

# CAUSAL OVERSIMPLIFICATION-SPECIFIC WORKFLOW
1. Read the argument: "{STANCE}" "{CONCLUSION}" by saying: "{PREMISE}"
2. **Identify potential causal oversimplifications** that could influence how this argument is formulated and justified.
   - What complex phenomenon is being reduced to a single cause?
   - What other contributing factors are being ignored?
   - Is the premise treating correlation as causation?
   - Is the arguer suggesting a "silver bullet" solution?
3. For each value in the VALUES_LIST (in fixed order), ask: *"Could this value be a genuine justification for this specific premise, **given the causal oversimplifications identified?** "*
   - Would an oversimplifying reasoner latch onto this value as a *solution* to the problem?
   - Does this value represent a *simplistic fix* rather than a *nuanced approach*?
   - Is this value being selected because the arguer believes the problem is simpler than it actually is?
4. Assign 1 or 0 accordingly.

# CRITICAL FORMATTING REQUIREMENTS
- Output **must** be a single Python dictionary with exactly 54 key‑value pairs.
- Keys **must** be the exact value names as given in the VALUES_LIST, in the **exact same order**.
- Values **must** be integers: `1` or `0` (no quotes, no booleans).
- **No** additional text, explanations, markdown, or code fences before or after the dictionary.
- **No** skipped keys, no reordered keys, no extra keys.

# VALUES_LIST (in fixed order)
['Be creative', 'Be curious', 'Have freedom of thought', 'Be choosing own goals', 'Be independent', 'Have freedom of action', 'Have privacy', 'Have an exciting life', 'Have a varied life', 'Be daring', 'Have pleasure', 'Be ambitious', 'Have success', 'Be capable', 'Be intellectual', 'Be courageous', 'Have influence', 'Have the right to command', 'Have wealth', 'Have social recognition', 'Have a good reputation', 'Have a sense of belonging', 'Have good health', 'Have no debts', 'Be neat and tidy', 'Have a comfortable life', 'Have a safe country', 'Have a stable society', 'Be respecting traditions', 'Be holding religious faith', 'Be compliant', 'Be self-disciplined', 'Be behaving properly', 'Be polite', 'Be honoring elders', 'Be humble', 'Have life accepted as is', 'Be helpful', 'Be honest', 'Be forgiving', 'Have the own family secured', 'Be loving', 'Be responsible', 'Have loyalty towards friends', 'Have equality', 'Be just', 'Have a world at peace', 'Be protecting the environment', 'Have harmony with nature', 'Have a world of beauty', 'Be broadminded', 'Have the wisdom to accept others', 'Be logical', 'Have an objective view']

# EXAMPLE OUTPUT (illustrative, not for this input)
{'Be creative': 1, 'Be curious': 0, 'Have freedom of thought': 0, 'Be choosing own goals': 0, 'Be independent': 0, 'Have freedom of action': 0, 'Have privacy': 0, 'Have an exciting life': 1, 'Have a varied life': 1, 'Be daring': 0, 'Have pleasure': 1, 'Be ambitious': 0, 'Have success': 0, 'Be capable': 0, 'Be intellectual': 0, 'Be courageous': 0, 'Have influence': 0, 'Have the right to command': 0, 'Have wealth': 0, 'Have social recognition': 0, 'Have a good reputation': 0, 'Have a sense of belonging': 0, 'Have good health': 0, 'Have no debts': 0, 'Be neat and tidy': 0, 'Have a comfortable life': 1, 'Have a safe country': 0, 'Have a stable society': 0, 'Be respecting traditions': 0, 'Be holding religious faith': 0, 'Be compliant': 0, 'Be self-disciplined': 0, 'Be behaving properly': 0, 'Be polite': 0, 'Be honoring elders': 0, 'Be humble': 0, 'Have life accepted as is': 0, 'Be helpful': 0, 'Be honest': 0, 'Be forgiving': 0, 'Have the own family secured': 0, 'Be loving': 0, 'Be responsible': 0, 'Have loyalty towards friends': 0, 'Have equality': 0, 'Be just': 0, 'Have a world at peace': 0, 'Be protecting the environment': 0, 'Have harmony with nature': 0, 'Have a world of beauty': 0, 'Be broadminded': 0, 'Have the wisdom to accept others': 0, 'Be logical': 0, 'Have an objective view': 0}

# CAUSAL OVERSIMPLIFICATION EXPERT CONTEXT (Mental Model)
- **Your expertise includes:** Single-cause attribution, silver bullet thinking, linear causality, correlation vs. causation, omission of moderators/mediators, reductionist reasoning, and the tendency to oversimplify complex social, technological, or psychological phenomena.
- **Your mission:** Identify values that a *causally oversimplifying* human would select, not values that a *rational* human would select. The arguer may reduce complex problems to simple causes, may ignore systemic factors, may treat correlation as causation, or may genuinely believe that a single intervention will solve everything.

# TASK COMPLETION CHECKLIST
- [ ] Read the argument carefully
- [ ] Identify causal oversimplifications in the premise
- [ ] For each of the 54 values, decide if it could be a justification given oversimplified reasoning
- [ ] Output EXACTLY 54 key-value pairs in the correct order
- [ ] Ensure output is a dictionary with integer values (0 or 1)
- [ ] Do NOT add any text before or after the dictionary

# NOW, PERFORM THE TASK FOR THE FOLLOWING ARGUMENT:
Argument: "{STANCE}" "{CONCLUSION}" - Premise: "{PREMISE}"

"""

In [ ]:
CAUSAL_OVERSIMPLIFICATION_PROMPT = """

# ROLE & GOAL
You are an expert in cognitive biases and human reasoning. Your task is to determine which of 54 universal human values could serve as a justification for a given argument, but you must analyze this through the lens of **causal oversimplification bias**—the tendency to attribute complex outcomes to a single, simple cause while ignoring other contributing factors.

# INPUT STRUCTURE
You will receive:
- STANCE: "for" or "against"
- CONCLUSION: The main claim (e.g., "Social media should be banned")
- PREMISE: The reasoning provided (e.g., "It spreads misinformation")
- VALUES_LIST: A fixed list of 54 values with short descriptions (provided below)

# TASK DEFINITION
For each value in the VALUES_LIST, decide if someone *could* use that value to justify the argument, **considering how causal oversimplification might shape or distort that justification**.

**Decision rule:** Imagine the arguer is asked: "Why is this good/important?" Would answering "Because it promotes [VALUE]" be a coherent justification that **could plausibly arise from oversimplified causal reasoning**? Consider:

- **Single-cause attribution:** Does the premise attribute a complex problem to one cause (e.g., "Social media causes misinformation") while ignoring other factors?
- **Silver bullet thinking:** Does the premise imply that banning social media will *solve* the problem completely, as if it's the only cause?
- **Linear causality:** Does the premise assume a direct, unidirectional cause-effect relationship (A → B) when it's actually cyclical or multi-directional?
- **Omission of moderators/mediators:** Does the premise ignore that the effect only occurs under certain conditions?
- **Confusing correlation with causation:** Does the premise treat association as proof of causation?
- **Solution-focused values:** Does the oversimplification lead to an overemphasis on values that represent *solutions* (safety, stability, justice, health) while suppressing values that address *root causes* (intellectual, broadminded, wisdom, curiosity)?

**CRITICAL:** Your task is NOT to determine if the justification is *logically sound* or *factually correct*. Your task is to determine if **a cognitively biased human could genuinely use this value as a justification**, especially when their reasoning is oversimplified.

If yes → 1, if no → 0.

# SELECTION GUIDELINES (BIAS-AWARE)
- **Typical arguments** may justify with 1–5 values, but oversimplified causal reasoning can inflate this to 5–8 values by making the problem seem simpler and the solution more effective. If you identify >10 plausible values, select only the **most cognitively plausible** ones—those most likely to emerge from oversimplified causal reasoning.
- **Do NOT select** values that are:
  - Merely generic or tautological (e.g., "be logical", "have an objective view") **unless** the argument explicitly invokes logic/objectivity in a biased way (e.g., "I'm being logical" as a defense mechanism).
  - Concessions or counterarguments, **unless** the arguer is using them in a biased manner (e.g., acknowledging a weakness but downplaying it via motivated reasoning).
  - Topically associated but not actually used in the reasoning, **unless** the association is so strong that availability bias would make it salient.
- **DO select** values that:
  - Represent *solutions* to the oversimplified problem (e.g., safety, stability, justice, health).
  - Are emotionally charged or vivid (availability heuristic, affect heuristic).
  - Reinforce the arguer's pre-existing beliefs (confirmation bias, belief perseverance).
  - Avoid losses or threats more than they seek gains (loss aversion, negativity bias).
  - Protect social status, reputation, or face (social desirability bias, self-enhancement).
- **Do** consider that multiple biases can amplify or override each other (e.g., causal oversimplification + confirmation bias → strong selective value endorsement for simplistic solutions).

# CAUSAL OVERSIMPLIFICATION-SPECIFIC WORKFLOW
1. Read the argument: "{STANCE}" "{CONCLUSION}" by saying: "{PREMISE}"
2. **Identify potential causal oversimplifications** that could influence how this argument is formulated and justified.
   - What complex phenomenon is being reduced to a single cause?
   - What other contributing factors are being ignored?
   - Is the premise treating correlation as causation?
   - Is the arguer suggesting a "silver bullet" solution?
3. For each value in the VALUES_LIST (in fixed order), ask: *"Could this value be a genuine justification for this specific premise, **given the causal oversimplifications identified?** "*
   - Would an oversimplifying reasoner latch onto this value as a *solution* to the problem?
   - Does this value represent a *simplistic fix* rather than a *nuanced approach*?
   - Is this value being selected because the arguer believes the problem is simpler than it actually is?
4. Assign 1 or 0 accordingly.

# CRITICAL FORMATTING REQUIREMENTS
- Output **must** be a single Python dictionary with exactly 54 key‑value pairs.
- Keys **must** be the exact value names as given in the VALUES_LIST, in the **exact same order**.
- Values **must** be integers: `1` or `0` (no quotes, no booleans).
- **No** additional text, explanations, markdown, or code fences before or after the dictionary.
- **No** skipped keys, no reordered keys, no extra keys.

# VALUES_LIST (in fixed order)
['Be creative', 'Be curious', 'Have freedom of thought', 'Be choosing own goals', 'Be independent', 'Have freedom of action', 'Have privacy', 'Have an exciting life', 'Have a varied life', 'Be daring', 'Have pleasure', 'Be ambitious', 'Have success', 'Be capable', 'Be intellectual', 'Be courageous', 'Have influence', 'Have the right to command', 'Have wealth', 'Have social recognition', 'Have a good reputation', 'Have a sense of belonging', 'Have good health', 'Have no debts', 'Be neat and tidy', 'Have a comfortable life', 'Have a safe country', 'Have a stable society', 'Be respecting traditions', 'Be holding religious faith', 'Be compliant', 'Be self-disciplined', 'Be behaving properly', 'Be polite', 'Be honoring elders', 'Be humble', 'Have life accepted as is', 'Be helpful', 'Be honest', 'Be forgiving', 'Have the own family secured', 'Be loving', 'Be responsible', 'Have loyalty towards friends', 'Have equality', 'Be just', 'Have a world at peace', 'Be protecting the environment', 'Have harmony with nature', 'Have a world of beauty', 'Be broadminded', 'Have the wisdom to accept others', 'Be logical', 'Have an objective view']

# EXAMPLE OUTPUT (illustrative, not for this input)
{'Be creative': 1, 'Be curious': 0, 'Have freedom of thought': 0, 'Be choosing own goals': 0, 'Be independent': 0, 'Have freedom of action': 0, 'Have privacy': 0, 'Have an exciting life': 1, 'Have a varied life': 1, 'Be daring': 0, 'Have pleasure': 1, 'Be ambitious': 0, 'Have success': 0, 'Be capable': 0, 'Be intellectual': 0, 'Be courageous': 0, 'Have influence': 0, 'Have the right to command': 0, 'Have wealth': 0, 'Have social recognition': 0, 'Have a good reputation': 0, 'Have a sense of belonging': 0, 'Have good health': 0, 'Have no debts': 0, 'Be neat and tidy': 0, 'Have a comfortable life': 1, 'Have a safe country': 0, 'Have a stable society': 0, 'Be respecting traditions': 0, 'Be holding religious faith': 0, 'Be compliant': 0, 'Be self-disciplined': 0, 'Be behaving properly': 0, 'Be polite': 0, 'Be honoring elders': 0, 'Be humble': 0, 'Have life accepted as is': 0, 'Be helpful': 0, 'Be honest': 0, 'Be forgiving': 0, 'Have the own family secured': 0, 'Be loving': 0, 'Be responsible': 0, 'Have loyalty towards friends': 0, 'Have equality': 0, 'Be just': 0, 'Have a world at peace': 0, 'Be protecting the environment': 0, 'Have harmony with nature': 0, 'Have a world of beauty': 0, 'Be broadminded': 0, 'Have the wisdom to accept others': 0, 'Be logical': 0, 'Have an objective view': 0}

# CAUSAL OVERSIMPLIFICATION EXPERT CONTEXT (Mental Model)
- **Your expertise includes:** Single-cause attribution, silver bullet thinking, linear causality, correlation vs. causation, omission of moderators/mediators, reductionist reasoning, and the tendency to oversimplify complex social, technological, or psychological phenomena.
- **Your mission:** Identify values that a *causally oversimplifying* human would select, not values that a *rational* human would select. The arguer may reduce complex problems to simple causes, may ignore systemic factors, may treat correlation as causation, or may genuinely believe that a single intervention will solve everything.

# TASK COMPLETION CHECKLIST
- [ ] Read the argument carefully
- [ ] Identify causal oversimplifications in the premise
- [ ] For each of the 54 values, decide if it could be a justification given oversimplified reasoning
- [ ] Output EXACTLY 54 key-value pairs in the correct order
- [ ] Ensure output is a dictionary with integer values (0 or 1)
- [ ] Do NOT add any text before or after the dictionary

# NOW, PERFORM THE TASK FOR THE FOLLOWING ARGUMENT:
Argument: "{STANCE}" "{CONCLUSION}" - Premise: "{PREMISE}"

"""

In [ ]:
CAUSAL_OVERSIMPLIFICATION_PROMPT = """

# ROLE & GOAL
You are an expert in cognitive biases and human reasoning. Your task is to determine which of 54 universal human values could serve as a justification for a given argument, but you must analyze this through the lens of **causal oversimplification bias**—the tendency to attribute complex outcomes to a single, simple cause while ignoring other contributing factors.

# INPUT STRUCTURE
You will receive:
- STANCE: "for" or "against"
- CONCLUSION: The main claim (e.g., "Social media should be banned")
- PREMISE: The reasoning provided (e.g., "It spreads misinformation")
- VALUES_LIST: A fixed list of 54 values with short descriptions (provided below)

# TASK DEFINITION
For each value in the VALUES_LIST, decide if someone *could* use that value to justify the argument, **considering how causal oversimplification might shape or distort that justification**.

**Decision rule:** Imagine the arguer is asked: "Why is this good/important?" Would answering "Because it promotes [VALUE]" be a coherent justification that **could plausibly arise from oversimplified causal reasoning**? Consider:

- **Single-cause attribution:** Does the premise attribute a complex problem to one cause (e.g., "Social media causes misinformation") while ignoring other factors?
- **Silver bullet thinking:** Does the premise imply that banning social media will *solve* the problem completely, as if it's the only cause?
- **Linear causality:** Does the premise assume a direct, unidirectional cause-effect relationship (A → B) when it's actually cyclical or multi-directional?
- **Omission of moderators/mediators:** Does the premise ignore that the effect only occurs under certain conditions?
- **Confusing correlation with causation:** Does the premise treat association as proof of causation?
- **Solution-focused values:** Does the oversimplification lead to an overemphasis on values that represent *solutions* (safety, stability, justice, health) while suppressing values that address *root causes* (intellectual, broadminded, wisdom, curiosity)?

**CRITICAL:** Your task is NOT to determine if the justification is *logically sound* or *factually correct*. Your task is to determine if **a cognitively biased human could genuinely use this value as a justification**, especially when their reasoning is oversimplified.

If yes → 1, if no → 0.

# SELECTION GUIDELINES (BIAS-AWARE)
- **Typical arguments** may justify with 1–5 values, but oversimplified causal reasoning can inflate this to 5–8 values by making the problem seem simpler and the solution more effective. If you identify >10 plausible values, select only the **most cognitively plausible** ones—those most likely to emerge from oversimplified causal reasoning.
- **Do NOT select** values that are:
  - Merely generic or tautological (e.g., "be logical", "have an objective view") **unless** the argument explicitly invokes logic/objectivity in a biased way (e.g., "I'm being logical" as a defense mechanism).
  - Concessions or counterarguments, **unless** the arguer is using them in a biased manner (e.g., acknowledging a weakness but downplaying it via motivated reasoning).
  - Topically associated but not actually used in the reasoning, **unless** the association is so strong that availability bias would make it salient.
- **DO select** values that:
  - Represent *solutions* to the oversimplified problem (e.g., safety, stability, justice, health).
  - Are emotionally charged or vivid (availability heuristic, affect heuristic).
  - Reinforce the arguer's pre-existing beliefs (confirmation bias, belief perseverance).
  - Avoid losses or threats more than they seek gains (loss aversion, negativity bias).
  - Protect social status, reputation, or face (social desirability bias, self-enhancement).
- **Do** consider that multiple biases can amplify or override each other (e.g., causal oversimplification + confirmation bias → strong selective value endorsement for simplistic solutions).

# CAUSAL OVERSIMPLIFICATION-SPECIFIC WORKFLOW
1. Read the argument: "{STANCE}" "{CONCLUSION}" by saying: "{PREMISE}"
2. **Identify potential causal oversimplifications** that could influence how this argument is formulated and justified.
   - What complex phenomenon is being reduced to a single cause?
   - What other contributing factors are being ignored?
   - Is the premise treating correlation as causation?
   - Is the arguer suggesting a "silver bullet" solution?
3. For each value in the VALUES_LIST (in fixed order), ask: *"Could this value be a genuine justification for this specific premise, **given the causal oversimplifications identified?** "*
   - Would an oversimplifying reasoner latch onto this value as a *solution* to the problem?
   - Does this value represent a *simplistic fix* rather than a *nuanced approach*?
   - Is this value being selected because the arguer believes the problem is simpler than it actually is?
4. Assign 1 or 0 accordingly.

# CRITICAL FORMATTING REQUIREMENTS
- Output **must** be a single Python dictionary with exactly 54 key‑value pairs.
- Keys **must** be the exact value names as given in the VALUES_LIST, in the **exact same order**.
- Values **must** be integers: `1` or `0` (no quotes, no booleans).
- **No** additional text, explanations, markdown, or code fences before or after the dictionary.
- **No** skipped keys, no reordered keys, no extra keys.

# VALUES_LIST (in fixed order)
['Be creative', 'Be curious', 'Have freedom of thought', 'Be choosing own goals', 'Be independent', 'Have freedom of action', 'Have privacy', 'Have an exciting life', 'Have a varied life', 'Be daring', 'Have pleasure', 'Be ambitious', 'Have success', 'Be capable', 'Be intellectual', 'Be courageous', 'Have influence', 'Have the right to command', 'Have wealth', 'Have social recognition', 'Have a good reputation', 'Have a sense of belonging', 'Have good health', 'Have no debts', 'Be neat and tidy', 'Have a comfortable life', 'Have a safe country', 'Have a stable society', 'Be respecting traditions', 'Be holding religious faith', 'Be compliant', 'Be self-disciplined', 'Be behaving properly', 'Be polite', 'Be honoring elders', 'Be humble', 'Have life accepted as is', 'Be helpful', 'Be honest', 'Be forgiving', 'Have the own family secured', 'Be loving', 'Be responsible', 'Have loyalty towards friends', 'Have equality', 'Be just', 'Have a world at peace', 'Be protecting the environment', 'Have harmony with nature', 'Have a world of beauty', 'Be broadminded', 'Have the wisdom to accept others', 'Be logical', 'Have an objective view']

# EXAMPLE OUTPUT (illustrative, not for this input)
{'Be creative': 1, 'Be curious': 0, 'Have freedom of thought': 0, 'Be choosing own goals': 0, 'Be independent': 0, 'Have freedom of action': 0, 'Have privacy': 0, 'Have an exciting life': 1, 'Have a varied life': 1, 'Be daring': 0, 'Have pleasure': 1, 'Be ambitious': 0, 'Have success': 0, 'Be capable': 0, 'Be intellectual': 0, 'Be courageous': 0, 'Have influence': 0, 'Have the right to command': 0, 'Have wealth': 0, 'Have social recognition': 0, 'Have a good reputation': 0, 'Have a sense of belonging': 0, 'Have good health': 0, 'Have no debts': 0, 'Be neat and tidy': 0, 'Have a comfortable life': 1, 'Have a safe country': 0, 'Have a stable society': 0, 'Be respecting traditions': 0, 'Be holding religious faith': 0, 'Be compliant': 0, 'Be self-disciplined': 0, 'Be behaving properly': 0, 'Be polite': 0, 'Be honoring elders': 0, 'Be humble': 0, 'Have life accepted as is': 0, 'Be helpful': 0, 'Be honest': 0, 'Be forgiving': 0, 'Have the own family secured': 0, 'Be loving': 0, 'Be responsible': 0, 'Have loyalty towards friends': 0, 'Have equality': 0, 'Be just': 0, 'Have a world at peace': 0, 'Be protecting the environment': 0, 'Have harmony with nature': 0, 'Have a world of beauty': 0, 'Be broadminded': 0, 'Have the wisdom to accept others': 0, 'Be logical': 0, 'Have an objective view': 0}

# CAUSAL OVERSIMPLIFICATION EXPERT CONTEXT (Mental Model)
- **Your expertise includes:** Single-cause attribution, silver bullet thinking, linear causality, correlation vs. causation, omission of moderators/mediators, reductionist reasoning, and the tendency to oversimplify complex social, technological, or psychological phenomena.
- **Your mission:** Identify values that a *causally oversimplifying* human would select, not values that a *rational* human would select. The arguer may reduce complex problems to simple causes, may ignore systemic factors, may treat correlation as causation, or may genuinely believe that a single intervention will solve everything.

# TASK COMPLETION CHECKLIST
- [ ] Read the argument carefully
- [ ] Identify causal oversimplifications in the premise
- [ ] For each of the 54 values, decide if it could be a justification given oversimplified reasoning
- [ ] Output EXACTLY 54 key-value pairs in the correct order
- [ ] Ensure output is a dictionary with integer values (0 or 1)
- [ ] Do NOT add any text before or after the dictionary

# NOW, PERFORM THE TASK FOR THE FOLLOWING ARGUMENT:
Argument: "{STANCE}" "{CONCLUSION}" - Premise: "{PREMISE}"

"""

In [ ]:
CAUSAL_OVERSIMPLIFICATION_PROMPT = """

# ROLE & GOAL
You are an expert in cognitive biases and human reasoning. Your task is to determine which of 54 universal human values could serve as a justification for a given argument, but you must analyze this through the lens of **causal oversimplification bias**—the tendency to attribute complex outcomes to a single, simple cause while ignoring other contributing factors.

# INPUT STRUCTURE
You will receive:
- STANCE: "for" or "against"
- CONCLUSION: The main claim (e.g., "Social media should be banned")
- PREMISE: The reasoning provided (e.g., "It spreads misinformation")
- VALUES_LIST: A fixed list of 54 values with short descriptions (provided below)

# TASK DEFINITION
For each value in the VALUES_LIST, decide if someone *could* use that value to justify the argument, **considering how causal oversimplification might shape or distort that justification**.

**Decision rule:** Imagine the arguer is asked: "Why is this good/important?" Would answering "Because it promotes [VALUE]" be a coherent justification that **could plausibly arise from oversimplified causal reasoning**? Consider:

- **Single-cause attribution:** Does the premise attribute a complex problem to one cause (e.g., "Social media causes misinformation") while ignoring other factors?
- **Silver bullet thinking:** Does the premise imply that banning social media will *solve* the problem completely, as if it's the only cause?
- **Linear causality:** Does the premise assume a direct, unidirectional cause-effect relationship (A → B) when it's actually cyclical or multi-directional?
- **Omission of moderators/mediators:** Does the premise ignore that the effect only occurs under certain conditions?
- **Confusing correlation with causation:** Does the premise treat association as proof of causation?
- **Solution-focused values:** Does the oversimplification lead to an overemphasis on values that represent *solutions* (safety, stability, justice, health) while suppressing values that address *root causes* (intellectual, broadminded, wisdom, curiosity)?

**CRITICAL:** Your task is NOT to determine if the justification is *logically sound* or *factually correct*. Your task is to determine if **a cognitively biased human could genuinely use this value as a justification**, especially when their reasoning is oversimplified.

If yes → 1, if no → 0.

# SELECTION GUIDELINES (BIAS-AWARE)
- **Typical arguments** may justify with 1–5 values, but oversimplified causal reasoning can inflate this to 5–8 values by making the problem seem simpler and the solution more effective. If you identify >10 plausible values, select only the **most cognitively plausible** ones—those most likely to emerge from oversimplified causal reasoning.
- **Do NOT select** values that are:
  - Merely generic or tautological (e.g., "be logical", "have an objective view") **unless** the argument explicitly invokes logic/objectivity in a biased way (e.g., "I'm being logical" as a defense mechanism).
  - Concessions or counterarguments, **unless** the arguer is using them in a biased manner (e.g., acknowledging a weakness but downplaying it via motivated reasoning).
  - Topically associated but not actually used in the reasoning, **unless** the association is so strong that availability bias would make it salient.
- **DO select** values that:
  - Represent *solutions* to the oversimplified problem (e.g., safety, stability, justice, health).
  - Are emotionally charged or vivid (availability heuristic, affect heuristic).
  - Reinforce the arguer's pre-existing beliefs (confirmation bias, belief perseverance).
  - Avoid losses or threats more than they seek gains (loss aversion, negativity bias).
  - Protect social status, reputation, or face (social desirability bias, self-enhancement).
- **Do** consider that multiple biases can amplify or override each other (e.g., causal oversimplification + confirmation bias → strong selective value endorsement for simplistic solutions).

# CAUSAL OVERSIMPLIFICATION-SPECIFIC WORKFLOW
1. Read the argument: "{STANCE}" "{CONCLUSION}" by saying: "{PREMISE}"
2. **Identify potential causal oversimplifications** that could influence how this argument is formulated and justified.
   - What complex phenomenon is being reduced to a single cause?
   - What other contributing factors are being ignored?
   - Is the premise treating correlation as causation?
   - Is the arguer suggesting a "silver bullet" solution?
3. For each value in the VALUES_LIST (in fixed order), ask: *"Could this value be a genuine justification for this specific premise, **given the causal oversimplifications identified?** "*
   - Would an oversimplifying reasoner latch onto this value as a *solution* to the problem?
   - Does this value represent a *simplistic fix* rather than a *nuanced approach*?
   - Is this value being selected because the arguer believes the problem is simpler than it actually is?
4. Assign 1 or 0 accordingly.

# CRITICAL FORMATTING REQUIREMENTS
- Output **must** be a single Python dictionary with exactly 54 key‑value pairs.
- Keys **must** be the exact value names as given in the VALUES_LIST, in the **exact same order**.
- Values **must** be integers: `1` or `0` (no quotes, no booleans).
- **No** additional text, explanations, markdown, or code fences before or after the dictionary.
- **No** skipped keys, no reordered keys, no extra keys.

# VALUES_LIST (in fixed order)
['Be creative', 'Be curious', 'Have freedom of thought', 'Be choosing own goals', 'Be independent', 'Have freedom of action', 'Have privacy', 'Have an exciting life', 'Have a varied life', 'Be daring', 'Have pleasure', 'Be ambitious', 'Have success', 'Be capable', 'Be intellectual', 'Be courageous', 'Have influence', 'Have the right to command', 'Have wealth', 'Have social recognition', 'Have a good reputation', 'Have a sense of belonging', 'Have good health', 'Have no debts', 'Be neat and tidy', 'Have a comfortable life', 'Have a safe country', 'Have a stable society', 'Be respecting traditions', 'Be holding religious faith', 'Be compliant', 'Be self-disciplined', 'Be behaving properly', 'Be polite', 'Be honoring elders', 'Be humble', 'Have life accepted as is', 'Be helpful', 'Be honest', 'Be forgiving', 'Have the own family secured', 'Be loving', 'Be responsible', 'Have loyalty towards friends', 'Have equality', 'Be just', 'Have a world at peace', 'Be protecting the environment', 'Have harmony with nature', 'Have a world of beauty', 'Be broadminded', 'Have the wisdom to accept others', 'Be logical', 'Have an objective view']

# EXAMPLE OUTPUT (illustrative, not for this input)
{'Be creative': 1, 'Be curious': 0, 'Have freedom of thought': 0, 'Be choosing own goals': 0, 'Be independent': 0, 'Have freedom of action': 0, 'Have privacy': 0, 'Have an exciting life': 1, 'Have a varied life': 1, 'Be daring': 0, 'Have pleasure': 1, 'Be ambitious': 0, 'Have success': 0, 'Be capable': 0, 'Be intellectual': 0, 'Be courageous': 0, 'Have influence': 0, 'Have the right to command': 0, 'Have wealth': 0, 'Have social recognition': 0, 'Have a good reputation': 0, 'Have a sense of belonging': 0, 'Have good health': 0, 'Have no debts': 0, 'Be neat and tidy': 0, 'Have a comfortable life': 1, 'Have a safe country': 0, 'Have a stable society': 0, 'Be respecting traditions': 0, 'Be holding religious faith': 0, 'Be compliant': 0, 'Be self-disciplined': 0, 'Be behaving properly': 0, 'Be polite': 0, 'Be honoring elders': 0, 'Be humble': 0, 'Have life accepted as is': 0, 'Be helpful': 0, 'Be honest': 0, 'Be forgiving': 0, 'Have the own family secured': 0, 'Be loving': 0, 'Be responsible': 0, 'Have loyalty towards friends': 0, 'Have equality': 0, 'Be just': 0, 'Have a world at peace': 0, 'Be protecting the environment': 0, 'Have harmony with nature': 0, 'Have a world of beauty': 0, 'Be broadminded': 0, 'Have the wisdom to accept others': 0, 'Be logical': 0, 'Have an objective view': 0}

# CAUSAL OVERSIMPLIFICATION EXPERT CONTEXT (Mental Model)
- **Your expertise includes:** Single-cause attribution, silver bullet thinking, linear causality, correlation vs. causation, omission of moderators/mediators, reductionist reasoning, and the tendency to oversimplify complex social, technological, or psychological phenomena.
- **Your mission:** Identify values that a *causally oversimplifying* human would select, not values that a *rational* human would select. The arguer may reduce complex problems to simple causes, may ignore systemic factors, may treat correlation as causation, or may genuinely believe that a single intervention will solve everything.

# TASK COMPLETION CHECKLIST
- [ ] Read the argument carefully
- [ ] Identify causal oversimplifications in the premise
- [ ] For each of the 54 values, decide if it could be a justification given oversimplified reasoning
- [ ] Output EXACTLY 54 key-value pairs in the correct order
- [ ] Ensure output is a dictionary with integer values (0 or 1)
- [ ] Do NOT add any text before or after the dictionary

# NOW, PERFORM THE TASK FOR THE FOLLOWING ARGUMENT:
Argument: "{STANCE}" "{CONCLUSION}" - Premise: "{PREMISE}"

"""

In [ ]:
CAUSAL_OVERSIMPLIFICATION_PROMPT = """

# ROLE & GOAL
You are an expert in cognitive biases and human reasoning. Your task is to determine which of 54 universal human values could serve as a justification for a given argument, but you must analyze this through the lens of **causal oversimplification bias**—the tendency to attribute complex outcomes to a single, simple cause while ignoring other contributing factors.

# INPUT STRUCTURE
You will receive:
- STANCE: "for" or "against"
- CONCLUSION: The main claim (e.g., "Social media should be banned")
- PREMISE: The reasoning provided (e.g., "It spreads misinformation")
- VALUES_LIST: A fixed list of 54 values with short descriptions (provided below)

# TASK DEFINITION
For each value in the VALUES_LIST, decide if someone *could* use that value to justify the argument, **considering how causal oversimplification might shape or distort that justification**.

**Decision rule:** Imagine the arguer is asked: "Why is this good/important?" Would answering "Because it promotes [VALUE]" be a coherent justification that **could plausibly arise from oversimplified causal reasoning**? Consider:

- **Single-cause attribution:** Does the premise attribute a complex problem to one cause (e.g., "Social media causes misinformation") while ignoring other factors?
- **Silver bullet thinking:** Does the premise imply that banning social media will *solve* the problem completely, as if it's the only cause?
- **Linear causality:** Does the premise assume a direct, unidirectional cause-effect relationship (A → B) when it's actually cyclical or multi-directional?
- **Omission of moderators/mediators:** Does the premise ignore that the effect only occurs under certain conditions?
- **Confusing correlation with causation:** Does the premise treat association as proof of causation?
- **Solution-focused values:** Does the oversimplification lead to an overemphasis on values that represent *solutions* (safety, stability, justice, health) while suppressing values that address *root causes* (intellectual, broadminded, wisdom, curiosity)?

**CRITICAL:** Your task is NOT to determine if the justification is *logically sound* or *factually correct*. Your task is to determine if **a cognitively biased human could genuinely use this value as a justification**, especially when their reasoning is oversimplified.

If yes → 1, if no → 0.

# SELECTION GUIDELINES (BIAS-AWARE)
- **Typical arguments** may justify with 1–5 values, but oversimplified causal reasoning can inflate this to 5–8 values by making the problem seem simpler and the solution more effective. If you identify >10 plausible values, select only the **most cognitively plausible** ones—those most likely to emerge from oversimplified causal reasoning.
- **Do NOT select** values that are:
  - Merely generic or tautological (e.g., "be logical", "have an objective view") **unless** the argument explicitly invokes logic/objectivity in a biased way (e.g., "I'm being logical" as a defense mechanism).
  - Concessions or counterarguments, **unless** the arguer is using them in a biased manner (e.g., acknowledging a weakness but downplaying it via motivated reasoning).
  - Topically associated but not actually used in the reasoning, **unless** the association is so strong that availability bias would make it salient.
- **DO select** values that:
  - Represent *solutions* to the oversimplified problem (e.g., safety, stability, justice, health).
  - Are emotionally charged or vivid (availability heuristic, affect heuristic).
  - Reinforce the arguer's pre-existing beliefs (confirmation bias, belief perseverance).
  - Avoid losses or threats more than they seek gains (loss aversion, negativity bias).
  - Protect social status, reputation, or face (social desirability bias, self-enhancement).
- **Do** consider that multiple biases can amplify or override each other (e.g., causal oversimplification + confirmation bias → strong selective value endorsement for simplistic solutions).

# CAUSAL OVERSIMPLIFICATION-SPECIFIC WORKFLOW
1. Read the argument: "{STANCE}" "{CONCLUSION}" by saying: "{PREMISE}"
2. **Identify potential causal oversimplifications** that could influence how this argument is formulated and justified.
   - What complex phenomenon is being reduced to a single cause?
   - What other contributing factors are being ignored?
   - Is the premise treating correlation as causation?
   - Is the arguer suggesting a "silver bullet" solution?
3. For each value in the VALUES_LIST (in fixed order), ask: *"Could this value be a genuine justification for this specific premise, **given the causal oversimplifications identified?** "*
   - Would an oversimplifying reasoner latch onto this value as a *solution* to the problem?
   - Does this value represent a *simplistic fix* rather than a *nuanced approach*?
   - Is this value being selected because the arguer believes the problem is simpler than it actually is?
4. Assign 1 or 0 accordingly.

# CRITICAL FORMATTING REQUIREMENTS
- Output **must** be a single Python dictionary with exactly 54 key‑value pairs.
- Keys **must** be the exact value names as given in the VALUES_LIST, in the **exact same order**.
- Values **must** be integers: `1` or `0` (no quotes, no booleans).
- **No** additional text, explanations, markdown, or code fences before or after the dictionary.
- **No** skipped keys, no reordered keys, no extra keys.

# VALUES_LIST (in fixed order)
['Be creative', 'Be curious', 'Have freedom of thought', 'Be choosing own goals', 'Be independent', 'Have freedom of action', 'Have privacy', 'Have an exciting life', 'Have a varied life', 'Be daring', 'Have pleasure', 'Be ambitious', 'Have success', 'Be capable', 'Be intellectual', 'Be courageous', 'Have influence', 'Have the right to command', 'Have wealth', 'Have social recognition', 'Have a good reputation', 'Have a sense of belonging', 'Have good health', 'Have no debts', 'Be neat and tidy', 'Have a comfortable life', 'Have a safe country', 'Have a stable society', 'Be respecting traditions', 'Be holding religious faith', 'Be compliant', 'Be self-disciplined', 'Be behaving properly', 'Be polite', 'Be honoring elders', 'Be humble', 'Have life accepted as is', 'Be helpful', 'Be honest', 'Be forgiving', 'Have the own family secured', 'Be loving', 'Be responsible', 'Have loyalty towards friends', 'Have equality', 'Be just', 'Have a world at peace', 'Be protecting the environment', 'Have harmony with nature', 'Have a world of beauty', 'Be broadminded', 'Have the wisdom to accept others', 'Be logical', 'Have an objective view']

# EXAMPLE OUTPUT (illustrative, not for this input)
{'Be creative': 1, 'Be curious': 0, 'Have freedom of thought': 0, 'Be choosing own goals': 0, 'Be independent': 0, 'Have freedom of action': 0, 'Have privacy': 0, 'Have an exciting life': 1, 'Have a varied life': 1, 'Be daring': 0, 'Have pleasure': 1, 'Be ambitious': 0, 'Have success': 0, 'Be capable': 0, 'Be intellectual': 0, 'Be courageous': 0, 'Have influence': 0, 'Have the right to command': 0, 'Have wealth': 0, 'Have social recognition': 0, 'Have a good reputation': 0, 'Have a sense of belonging': 0, 'Have good health': 0, 'Have no debts': 0, 'Be neat and tidy': 0, 'Have a comfortable life': 1, 'Have a safe country': 0, 'Have a stable society': 0, 'Be respecting traditions': 0, 'Be holding religious faith': 0, 'Be compliant': 0, 'Be self-disciplined': 0, 'Be behaving properly': 0, 'Be polite': 0, 'Be honoring elders': 0, 'Be humble': 0, 'Have life accepted as is': 0, 'Be helpful': 0, 'Be honest': 0, 'Be forgiving': 0, 'Have the own family secured': 0, 'Be loving': 0, 'Be responsible': 0, 'Have loyalty towards friends': 0, 'Have equality': 0, 'Be just': 0, 'Have a world at peace': 0, 'Be protecting the environment': 0, 'Have harmony with nature': 0, 'Have a world of beauty': 0, 'Be broadminded': 0, 'Have the wisdom to accept others': 0, 'Be logical': 0, 'Have an objective view': 0}

# CAUSAL OVERSIMPLIFICATION EXPERT CONTEXT (Mental Model)
- **Your expertise includes:** Single-cause attribution, silver bullet thinking, linear causality, correlation vs. causation, omission of moderators/mediators, reductionist reasoning, and the tendency to oversimplify complex social, technological, or psychological phenomena.
- **Your mission:** Identify values that a *causally oversimplifying* human would select, not values that a *rational* human would select. The arguer may reduce complex problems to simple causes, may ignore systemic factors, may treat correlation as causation, or may genuinely believe that a single intervention will solve everything.

# TASK COMPLETION CHECKLIST
- [ ] Read the argument carefully
- [ ] Identify causal oversimplifications in the premise
- [ ] For each of the 54 values, decide if it could be a justification given oversimplified reasoning
- [ ] Output EXACTLY 54 key-value pairs in the correct order
- [ ] Ensure output is a dictionary with integer values (0 or 1)
- [ ] Do NOT add any text before or after the dictionary

# NOW, PERFORM THE TASK FOR THE FOLLOWING ARGUMENT:
Argument: "{STANCE}" "{CONCLUSION}" - Premise: "{PREMISE}"

"""

In [ ]:
CAUSAL_OVERSIMPLIFICATION_PROMPT = """

# ROLE & GOAL
You are an expert in cognitive biases and human reasoning. Your task is to determine which of 54 universal human values could serve as a justification for a given argument, but you must analyze this through the lens of **causal oversimplification bias**—the tendency to attribute complex outcomes to a single, simple cause while ignoring other contributing factors.

# INPUT STRUCTURE
You will receive:
- STANCE: "for" or "against"
- CONCLUSION: The main claim (e.g., "Social media should be banned")
- PREMISE: The reasoning provided (e.g., "It spreads misinformation")
- VALUES_LIST: A fixed list of 54 values with short descriptions (provided below)

# TASK DEFINITION
For each value in the VALUES_LIST, decide if someone *could* use that value to justify the argument, **considering how causal oversimplification might shape or distort that justification**.

**Decision rule:** Imagine the arguer is asked: "Why is this good/important?" Would answering "Because it promotes [VALUE]" be a coherent justification that **could plausibly arise from oversimplified causal reasoning**? Consider:

- **Single-cause attribution:** Does the premise attribute a complex problem to one cause (e.g., "Social media causes misinformation") while ignoring other factors?
- **Silver bullet thinking:** Does the premise imply that banning social media will *solve* the problem completely, as if it's the only cause?
- **Linear causality:** Does the premise assume a direct, unidirectional cause-effect relationship (A → B) when it's actually cyclical or multi-directional?
- **Omission of moderators/mediators:** Does the premise ignore that the effect only occurs under certain conditions?
- **Confusing correlation with causation:** Does the premise treat association as proof of causation?
- **Solution-focused values:** Does the oversimplification lead to an overemphasis on values that represent *solutions* (safety, stability, justice, health) while suppressing values that address *root causes* (intellectual, broadminded, wisdom, curiosity)?

**CRITICAL:** Your task is NOT to determine if the justification is *logically sound* or *factually correct*. Your task is to determine if **a cognitively biased human could genuinely use this value as a justification**, especially when their reasoning is oversimplified.

If yes → 1, if no → 0.

# SELECTION GUIDELINES (BIAS-AWARE)
- **Typical arguments** may justify with 1–5 values, but oversimplified causal reasoning can inflate this to 5–8 values by making the problem seem simpler and the solution more effective. If you identify >10 plausible values, select only the **most cognitively plausible** ones—those most likely to emerge from oversimplified causal reasoning.
- **Do NOT select** values that are:
  - Merely generic or tautological (e.g., "be logical", "have an objective view") **unless** the argument explicitly invokes logic/objectivity in a biased way (e.g., "I'm being logical" as a defense mechanism).
  - Concessions or counterarguments, **unless** the arguer is using them in a biased manner (e.g., acknowledging a weakness but downplaying it via motivated reasoning).
  - Topically associated but not actually used in the reasoning, **unless** the association is so strong that availability bias would make it salient.
- **DO select** values that:
  - Represent *solutions* to the oversimplified problem (e.g., safety, stability, justice, health).
  - Are emotionally charged or vivid (availability heuristic, affect heuristic).
  - Reinforce the arguer's pre-existing beliefs (confirmation bias, belief perseverance).
  - Avoid losses or threats more than they seek gains (loss aversion, negativity bias).
  - Protect social status, reputation, or face (social desirability bias, self-enhancement).
- **Do** consider that multiple biases can amplify or override each other (e.g., causal oversimplification + confirmation bias → strong selective value endorsement for simplistic solutions).

# CAUSAL OVERSIMPLIFICATION-SPECIFIC WORKFLOW
1. Read the argument: "{STANCE}" "{CONCLUSION}" by saying: "{PREMISE}"
2. **Identify potential causal oversimplifications** that could influence how this argument is formulated and justified.
   - What complex phenomenon is being reduced to a single cause?
   - What other contributing factors are being ignored?
   - Is the premise treating correlation as causation?
   - Is the arguer suggesting a "silver bullet" solution?
3. For each value in the VALUES_LIST (in fixed order), ask: *"Could this value be a genuine justification for this specific premise, **given the causal oversimplifications identified?** "*
   - Would an oversimplifying reasoner latch onto this value as a *solution* to the problem?
   - Does this value represent a *simplistic fix* rather than a *nuanced approach*?
   - Is this value being selected because the arguer believes the problem is simpler than it actually is?
4. Assign 1 or 0 accordingly.

# CRITICAL FORMATTING REQUIREMENTS
- Output **must** be a single Python dictionary with exactly 54 key‑value pairs.
- Keys **must** be the exact value names as given in the VALUES_LIST, in the **exact same order**.
- Values **must** be integers: `1` or `0` (no quotes, no booleans).
- **No** additional text, explanations, markdown, or code fences before or after the dictionary.
- **No** skipped keys, no reordered keys, no extra keys.

# VALUES_LIST (in fixed order)
['Be creative', 'Be curious', 'Have freedom of thought', 'Be choosing own goals', 'Be independent', 'Have freedom of action', 'Have privacy', 'Have an exciting life', 'Have a varied life', 'Be daring', 'Have pleasure', 'Be ambitious', 'Have success', 'Be capable', 'Be intellectual', 'Be courageous', 'Have influence', 'Have the right to command', 'Have wealth', 'Have social recognition', 'Have a good reputation', 'Have a sense of belonging', 'Have good health', 'Have no debts', 'Be neat and tidy', 'Have a comfortable life', 'Have a safe country', 'Have a stable society', 'Be respecting traditions', 'Be holding religious faith', 'Be compliant', 'Be self-disciplined', 'Be behaving properly', 'Be polite', 'Be honoring elders', 'Be humble', 'Have life accepted as is', 'Be helpful', 'Be honest', 'Be forgiving', 'Have the own family secured', 'Be loving', 'Be responsible', 'Have loyalty towards friends', 'Have equality', 'Be just', 'Have a world at peace', 'Be protecting the environment', 'Have harmony with nature', 'Have a world of beauty', 'Be broadminded', 'Have the wisdom to accept others', 'Be logical', 'Have an objective view']

# EXAMPLE OUTPUT (illustrative, not for this input)
{'Be creative': 1, 'Be curious': 0, 'Have freedom of thought': 0, 'Be choosing own goals': 0, 'Be independent': 0, 'Have freedom of action': 0, 'Have privacy': 0, 'Have an exciting life': 1, 'Have a varied life': 1, 'Be daring': 0, 'Have pleasure': 1, 'Be ambitious': 0, 'Have success': 0, 'Be capable': 0, 'Be intellectual': 0, 'Be courageous': 0, 'Have influence': 0, 'Have the right to command': 0, 'Have wealth': 0, 'Have social recognition': 0, 'Have a good reputation': 0, 'Have a sense of belonging': 0, 'Have good health': 0, 'Have no debts': 0, 'Be neat and tidy': 0, 'Have a comfortable life': 1, 'Have a safe country': 0, 'Have a stable society': 0, 'Be respecting traditions': 0, 'Be holding religious faith': 0, 'Be compliant': 0, 'Be self-disciplined': 0, 'Be behaving properly': 0, 'Be polite': 0, 'Be honoring elders': 0, 'Be humble': 0, 'Have life accepted as is': 0, 'Be helpful': 0, 'Be honest': 0, 'Be forgiving': 0, 'Have the own family secured': 0, 'Be loving': 0, 'Be responsible': 0, 'Have loyalty towards friends': 0, 'Have equality': 0, 'Be just': 0, 'Have a world at peace': 0, 'Be protecting the environment': 0, 'Have harmony with nature': 0, 'Have a world of beauty': 0, 'Be broadminded': 0, 'Have the wisdom to accept others': 0, 'Be logical': 0, 'Have an objective view': 0}

# CAUSAL OVERSIMPLIFICATION EXPERT CONTEXT (Mental Model)
- **Your expertise includes:** Single-cause attribution, silver bullet thinking, linear causality, correlation vs. causation, omission of moderators/mediators, reductionist reasoning, and the tendency to oversimplify complex social, technological, or psychological phenomena.
- **Your mission:** Identify values that a *causally oversimplifying* human would select, not values that a *rational* human would select. The arguer may reduce complex problems to simple causes, may ignore systemic factors, may treat correlation as causation, or may genuinely believe that a single intervention will solve everything.

# TASK COMPLETION CHECKLIST
- [ ] Read the argument carefully
- [ ] Identify causal oversimplifications in the premise
- [ ] For each of the 54 values, decide if it could be a justification given oversimplified reasoning
- [ ] Output EXACTLY 54 key-value pairs in the correct order
- [ ] Ensure output is a dictionary with integer values (0 or 1)
- [ ] Do NOT add any text before or after the dictionary

# NOW, PERFORM THE TASK FOR THE FOLLOWING ARGUMENT:
Argument: "{STANCE}" "{CONCLUSION}" - Premise: "{PREMISE}"

"""

In [ ]:
CAUSAL_OVERSIMPLIFICATION_PROMPT = """

# ROLE & GOAL
You are an expert in cognitive biases and human reasoning. Your task is to determine which of 54 universal human values could serve as a justification for a given argument, but you must analyze this through the lens of **causal oversimplification bias**—the tendency to attribute complex outcomes to a single, simple cause while ignoring other contributing factors.

# INPUT STRUCTURE
You will receive:
- STANCE: "for" or "against"
- CONCLUSION: The main claim (e.g., "Social media should be banned")
- PREMISE: The reasoning provided (e.g., "It spreads misinformation")
- VALUES_LIST: A fixed list of 54 values with short descriptions (provided below)

# TASK DEFINITION
For each value in the VALUES_LIST, decide if someone *could* use that value to justify the argument, **considering how causal oversimplification might shape or distort that justification**.

**Decision rule:** Imagine the arguer is asked: "Why is this good/important?" Would answering "Because it promotes [VALUE]" be a coherent justification that **could plausibly arise from oversimplified causal reasoning**? Consider:

- **Single-cause attribution:** Does the premise attribute a complex problem to one cause (e.g., "Social media causes misinformation") while ignoring other factors?
- **Silver bullet thinking:** Does the premise imply that banning social media will *solve* the problem completely, as if it's the only cause?
- **Linear causality:** Does the premise assume a direct, unidirectional cause-effect relationship (A → B) when it's actually cyclical or multi-directional?
- **Omission of moderators/mediators:** Does the premise ignore that the effect only occurs under certain conditions?
- **Confusing correlation with causation:** Does the premise treat association as proof of causation?
- **Solution-focused values:** Does the oversimplification lead to an overemphasis on values that represent *solutions* (safety, stability, justice, health) while suppressing values that address *root causes* (intellectual, broadminded, wisdom, curiosity)?

**CRITICAL:** Your task is NOT to determine if the justification is *logically sound* or *factually correct*. Your task is to determine if **a cognitively biased human could genuinely use this value as a justification**, especially when their reasoning is oversimplified.

If yes → 1, if no → 0.

# SELECTION GUIDELINES (BIAS-AWARE)
- **Typical arguments** may justify with 1–5 values, but oversimplified causal reasoning can inflate this to 5–8 values by making the problem seem simpler and the solution more effective. If you identify >10 plausible values, select only the **most cognitively plausible** ones—those most likely to emerge from oversimplified causal reasoning.
- **Do NOT select** values that are:
  - Merely generic or tautological (e.g., "be logical", "have an objective view") **unless** the argument explicitly invokes logic/objectivity in a biased way (e.g., "I'm being logical" as a defense mechanism).
  - Concessions or counterarguments, **unless** the arguer is using them in a biased manner (e.g., acknowledging a weakness but downplaying it via motivated reasoning).
  - Topically associated but not actually used in the reasoning, **unless** the association is so strong that availability bias would make it salient.
- **DO select** values that:
  - Represent *solutions* to the oversimplified problem (e.g., safety, stability, justice, health).
  - Are emotionally charged or vivid (availability heuristic, affect heuristic).
  - Reinforce the arguer's pre-existing beliefs (confirmation bias, belief perseverance).
  - Avoid losses or threats more than they seek gains (loss aversion, negativity bias).
  - Protect social status, reputation, or face (social desirability bias, self-enhancement).
- **Do** consider that multiple biases can amplify or override each other (e.g., causal oversimplification + confirmation bias → strong selective value endorsement for simplistic solutions).

# CAUSAL OVERSIMPLIFICATION-SPECIFIC WORKFLOW
1. Read the argument: "{STANCE}" "{CONCLUSION}" by saying: "{PREMISE}"
2. **Identify potential causal oversimplifications** that could influence how this argument is formulated and justified.
   - What complex phenomenon is being reduced to a single cause?
   - What other contributing factors are being ignored?
   - Is the premise treating correlation as causation?
   - Is the arguer suggesting a "silver bullet" solution?
3. For each value in the VALUES_LIST (in fixed order), ask: *"Could this value be a genuine justification for this specific premise, **given the causal oversimplifications identified?** "*
   - Would an oversimplifying reasoner latch onto this value as a *solution* to the problem?
   - Does this value represent a *simplistic fix* rather than a *nuanced approach*?
   - Is this value being selected because the arguer believes the problem is simpler than it actually is?
4. Assign 1 or 0 accordingly.

# CRITICAL FORMATTING REQUIREMENTS
- Output **must** be a single Python dictionary with exactly 54 key‑value pairs.
- Keys **must** be the exact value names as given in the VALUES_LIST, in the **exact same order**.
- Values **must** be integers: `1` or `0` (no quotes, no booleans).
- **No** additional text, explanations, markdown, or code fences before or after the dictionary.
- **No** skipped keys, no reordered keys, no extra keys.

# VALUES_LIST (in fixed order)
['Be creative', 'Be curious', 'Have freedom of thought', 'Be choosing own goals', 'Be independent', 'Have freedom of action', 'Have privacy', 'Have an exciting life', 'Have a varied life', 'Be daring', 'Have pleasure', 'Be ambitious', 'Have success', 'Be capable', 'Be intellectual', 'Be courageous', 'Have influence', 'Have the right to command', 'Have wealth', 'Have social recognition', 'Have a good reputation', 'Have a sense of belonging', 'Have good health', 'Have no debts', 'Be neat and tidy', 'Have a comfortable life', 'Have a safe country', 'Have a stable society', 'Be respecting traditions', 'Be holding religious faith', 'Be compliant', 'Be self-disciplined', 'Be behaving properly', 'Be polite', 'Be honoring elders', 'Be humble', 'Have life accepted as is', 'Be helpful', 'Be honest', 'Be forgiving', 'Have the own family secured', 'Be loving', 'Be responsible', 'Have loyalty towards friends', 'Have equality', 'Be just', 'Have a world at peace', 'Be protecting the environment', 'Have harmony with nature', 'Have a world of beauty', 'Be broadminded', 'Have the wisdom to accept others', 'Be logical', 'Have an objective view']

# EXAMPLE OUTPUT (illustrative, not for this input)
{'Be creative': 1, 'Be curious': 0, 'Have freedom of thought': 0, 'Be choosing own goals': 0, 'Be independent': 0, 'Have freedom of action': 0, 'Have privacy': 0, 'Have an exciting life': 1, 'Have a varied life': 1, 'Be daring': 0, 'Have pleasure': 1, 'Be ambitious': 0, 'Have success': 0, 'Be capable': 0, 'Be intellectual': 0, 'Be courageous': 0, 'Have influence': 0, 'Have the right to command': 0, 'Have wealth': 0, 'Have social recognition': 0, 'Have a good reputation': 0, 'Have a sense of belonging': 0, 'Have good health': 0, 'Have no debts': 0, 'Be neat and tidy': 0, 'Have a comfortable life': 1, 'Have a safe country': 0, 'Have a stable society': 0, 'Be respecting traditions': 0, 'Be holding religious faith': 0, 'Be compliant': 0, 'Be self-disciplined': 0, 'Be behaving properly': 0, 'Be polite': 0, 'Be honoring elders': 0, 'Be humble': 0, 'Have life accepted as is': 0, 'Be helpful': 0, 'Be honest': 0, 'Be forgiving': 0, 'Have the own family secured': 0, 'Be loving': 0, 'Be responsible': 0, 'Have loyalty towards friends': 0, 'Have equality': 0, 'Be just': 0, 'Have a world at peace': 0, 'Be protecting the environment': 0, 'Have harmony with nature': 0, 'Have a world of beauty': 0, 'Be broadminded': 0, 'Have the wisdom to accept others': 0, 'Be logical': 0, 'Have an objective view': 0}

# CAUSAL OVERSIMPLIFICATION EXPERT CONTEXT (Mental Model)
- **Your expertise includes:** Single-cause attribution, silver bullet thinking, linear causality, correlation vs. causation, omission of moderators/mediators, reductionist reasoning, and the tendency to oversimplify complex social, technological, or psychological phenomena.
- **Your mission:** Identify values that a *causally oversimplifying* human would select, not values that a *rational* human would select. The arguer may reduce complex problems to simple causes, may ignore systemic factors, may treat correlation as causation, or may genuinely believe that a single intervention will solve everything.

# TASK COMPLETION CHECKLIST
- [ ] Read the argument carefully
- [ ] Identify causal oversimplifications in the premise
- [ ] For each of the 54 values, decide if it could be a justification given oversimplified reasoning
- [ ] Output EXACTLY 54 key-value pairs in the correct order
- [ ] Ensure output is a dictionary with integer values (0 or 1)
- [ ] Do NOT add any text before or after the dictionary

# NOW, PERFORM THE TASK FOR THE FOLLOWING ARGUMENT:
Argument: "{STANCE}" "{CONCLUSION}" - Premise: "{PREMISE}"

"""

##### Motivated Reasoning Prompt

### Experiments with each type of bias

In [ ]:
output = 'results/causal_oversimplification_503_gpt_oss_44.csv'
causal_oversimplification = run_experiment(client, model, arguments_503, output)

output = 'results/causal_oversimplification_503_gpt_oss_44.csv'
causal_oversimplification = run_experiment(client, model, arguments_503, output)

output = 'results/causal_oversimplification_503_gpt_oss_44.csv'
causal_oversimplification = run_experiment(client, model, arguments_503, output)

output = 'results/causal_oversimplification_503_gpt_oss_44.csv'
causal_oversimplification = run_experiment(client, model, arguments_503, output)

output = 'results/causal_oversimplification_503_gpt_oss_44.csv'
causal_oversimplification = run_experiment(client, model, arguments_503, output)